In [ ]:
# Scrapping and crawling modules
from config import *

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google import genai
from dotenv import load_dotenv
from google.genai import types
import pydantic

import time
import re
import datetime
import os
from typing import *

import json
import os
import numpy as np

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import smtplib

import sqlite3

In [ ]:
# CONSTANTS
WEB_DICT = {
    "Disnakerja": "https://www.disnakerja.com/page/",
    "Inginkerja": "https://inginkerja.id/page/",
    "RekrutmenBersama": "https://rekrutmenbersama.co.id/page/"
    }

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GENAI_CLIENT = genai.Client(api_key=GEMINI_API_KEY)

SHORT_WAIT = 2  # used for short delays
LONG_WAIT = 8  # used for long delays
API_TIMEOUT = 30 # used for API calls timeout

MAX_PAST_DAYS = 14  # maximum number of days to consider a job posting as recent
CURRENT_DATE = datetime.datetime.now().date()
MAX_PAST_DATE = CURRENT_DATE - datetime.timedelta(days=MAX_PAST_DAYS)

# Database paths and table names
ACCEPTED_DB_PATH = "SQL_DATA/Accepted_vacancies.db"
TABLE_NAME_ACCEPTED_DB = "Accepted_vacancies"
OVERALL_DB_PATH = "SQL_DATA/All_logged_vacancies.db"
TABLE_NAME_OVERALL_DB = "All_vacancies"

CHROME_EXECUTABLE_PATH = "D:/chrome-win64/chrome.exe"
DRIVER_EXECUTABLE_PATH = "D:/chromedriver-win64/chromedriver.exe"

DRIVER = uc.Chrome(browser_executable_path=CHROME_EXECUTABLE_PATH, driver_executable_path=DRIVER_EXECUTABLE_PATH, headless=False)

# Load webhook if available
WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")
if WEBHOOK_URL:
    from discord import SyncWebhook, Embed, Color

def send_email(text: str = "", EMAIL_CREDENTIALS: dict = None):
    s = smtplib.SMTP('smtp.gmail.com', 587)
    s.starttls()
    s.login(EMAIL_CREDENTIALS["email"], EMAIL_CREDENTIALS["password"])

    message = MIMEMultipart("alternative")
    message["Subject"] = "VACANT BOT Error Notification"
    message["From"] = EMAIL_CREDENTIALS["email"]
    message["To"] = EMAIL_CREDENTIALS["send_to_email"]
    text = f"""\
    Hi, this is an automated email from your syntax :3.
    Some error has occurred in the vacant bot, please check the log for more details.
    {text}
    """

    part1 = MIMEText(text, "plain")
    message.attach(part1)
    s.sendmail(EMAIL_CREDENTIALS["email"], EMAIL_CREDENTIALS["send_to_email"], message.as_string())
    s.close()


# Discod Bot func ---------------
def send_post_on_discord(job: dict):
    try:
        # Initialize the synchronous webhook
        webhook = SyncWebhook.from_url(WEBHOOK_URL)
        
        position = job.get("Position", "New Job Opportunity")
        if isinstance(position, list):
            position = ", ".join(position) if position else "New Job Opportunity"

        reason = job.get("Reason", "No reason provided.")
        if len(reason) > 1024:
            reason = reason[:1021] + "..."

        embed = Embed(
            title=position,
            description=f"**{job.get('Nama_Perusahaan', 'Unknown Company')}**",
            color=Color.brand_green(),
        )

        if job.get("Icon"):
            embed.set_thumbnail(url=job["Icon"])

        embed.add_field(name="Location", value=job.get("Location", "N/A"), inline=True)
        embed.add_field(name="Experience", value=job.get("Experience_Needed", "N/A"), inline=True)
        embed.add_field(name="Last_Updated", value=job.get("Last_Updated", "N/A"), inline=True)
        embed.add_field(name="Views", value=job.get("View_Count", "N/A"), inline=True)
        embed.add_field(name="AI Match Reasoning", value=reason, inline=False)
        embed.add_field(name="Job Post Link", value=f"[Click Here]({job['Post_link']})", inline=False)
        
        # Send it directly
        webhook.send(embed=embed)

    except Exception as exc:
        print(f"Error sending to Discord: {exc}")


# Gemini function ---------------
def parse_json_response(response_text: str) -> dict:
    '''
    This function takes a JSON response text from the Gemini API and parses it into a dictionary.

    Parameters
    ----------
    response_text : str
        The JSON response text from the Gemini API.

    Returns
    ----------
    dict
        A dictionary containing the parsed response.
    '''
    response_text = response_text.text.strip()

    try:
        # Normal json load
        response_dict = json.loads(response_text)
    except json.JSONDecodeError:
        # If method above doesn't work.
        match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if match:
            try:
                response_dict = json.loads(match.group(0))
            except Exception as exc:
                response_dict = {}
        else:
            response_dict = {}

    # Now safely get your values (using .get() prevents KeyError if the schema fails)
    acceptance = response_dict.get("acceptance", "0")
    position = response_dict.get("position", [])
    reason = response_dict.get("reason", "No reason provided.")

    return acceptance, position, reason


class gemini_response_format(pydantic.BaseModel):
    acceptance: str
    position: list[str]
    reason: str


def verify_vacancy(input_text: str) -> tuple:
    '''
    This function takes an input text (applicant information) and sends it to the Gemini API for evaluation against the job description.

    Parameters
    ----------
    input_text : str
        The applicant's information to be evaluated.

    Returns
    ----------
    tuple
        A tuple containing the Gemini API response and a status code (1 for success, 0 for failure).
    '''

    status = None
    gemini_response = None

    # Use main model, if it fails, fallback to the secondary model
    for model in [MAIN_MODEL] + [FALLBACK_MODEL]:
        try:
            gemini_response = GENAI_CLIENT.models.generate_content(
                model=model,
                contents=[SYSTEM_PROMPT, input_text],
                config=types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=gemini_response_format,
                ))
            status = 1
            break
        except Exception as exc:
            status = 0

    return gemini_response, status

# Database functions ---------------
def init_db() -> None:
    '''
    Initialize the sqlite database and create table if it doesn't exist
    '''
    # Make this dir if no exists
    if not os.path.exists("SQL_DATA"):
        os.makedirs("SQL_DATA")

    # DB for accepted vacancies
    with sqlite3.connect(ACCEPTED_DB_PATH) as conn:
        conn.execute(
            f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME_ACCEPTED_DB} (
                ID INTEGER PRIMARY KEY AUTOINCREMENT,
                Nama_Perusahaan TEXT,
                View_Count TEXT,
                Last_Updated TEXT,
                Location TEXT,
                Experience_Needed TEXT,
                Position TEXT,
                Reason TEXT,
                Post_link TEXT,
                Website_Loker TEXT
            )
            """
        )

        # Links are unique and it should be unique, if not i'm retarded
        conn.execute(
            f"CREATE UNIQUE INDEX IF NOT EXISTS idx_{TABLE_NAME_ACCEPTED_DB}_link ON {TABLE_NAME_ACCEPTED_DB}(Post_link)"
        )

        conn.commit()

    # DB for all vacancies
    with sqlite3.connect(OVERALL_DB_PATH) as conn:
        conn.execute(
            f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME_OVERALL_DB} (
                ID INTEGER PRIMARY KEY AUTOINCREMENT,
                Nama_Perusahaan TEXT,
                View_Count TEXT,
                Last_Updated TEXT,
                Location TEXT,
                Experience_Needed TEXT,
                Position TEXT,
                Post_link TEXT,
                Website_Loker TEXT,
                DESCRIPTION TEXT
            )
            """
        )
        conn.execute(
            f"CREATE UNIQUE INDEX IF NOT EXISTS idx_{TABLE_NAME_OVERALL_DB}_link ON {TABLE_NAME_OVERALL_DB}(Post_link)"
        )

        conn.commit()


def insert_rows(rows: List[dict], db_type: Literal["accepted", "overall"]) -> int:
    '''
    Insert multiple rows into the database by appending them from the bottom
    
    Parameters
    ----------
    - rows : List[dict]
        A list of dictionaries representing the rows to be inserted
    - db_type : Literal["accepted", "overall"]
        The type of database to insert into, either "accepted" for accepted vacancies or "overall" for all logged vacancies
    Returns
    -------
    - int
        The number of rows inserted
    '''
    if not rows:
        return 0

    rows = rows[::-1] # Reverse the list to insert the bottom rows first

    match db_type:
        case "accepted":
            # Insert rows into the database
            with sqlite3.connect(ACCEPTED_DB_PATH) as conn:
            # Updated insert statement
                conn.executemany(
                    f"""
                    INSERT OR IGNORE INTO {TABLE_NAME_ACCEPTED_DB}
                    (Nama_Perusahaan, View_Count, Last_Updated, Location, Experience_Needed, Position, Reason, Post_link, Website_Loker)
                    VALUES (:Nama_Perusahaan, :View_Count, :Last_Updated, :Location, :Experience_Needed, :Position, :Reason, :Post_link, :Website_Loker)
                    """,
                    rows,
                )
                conn.commit()
        case "overall":
            with sqlite3.connect(OVERALL_DB_PATH) as conn:
                conn.executemany(
                    f"""
                    INSERT OR IGNORE INTO {TABLE_NAME_OVERALL_DB}
                    (Nama_Perusahaan, View_Count, Last_Updated, Location, Experience_Needed, Position, Post_link, Website_Loker, DESCRIPTION)
                    VALUES (:Nama_Perusahaan, :View_Count, :Last_Updated, :Location, :Experience_Needed, :Position, :Post_link, :Website_Loker, :DESCRIPTION)
                    """,
                    rows,
                )
                conn.commit()
    
    return conn.total_changes


def log_URLs(web_type : Literal["Disnakerja", "Inginkerja", "RekrutmenBersama"], url : list[str]) -> None:
    '''
    Log the URLs that have been processed to avoid re-scraping in the future

    Parameters
    ----------
    - web_type : Literal["Disnakerja", "Inginkerja", "RekrutmenBersama"]
        The type of job website to log the URLs for
    - url : list[str]
        A list of URLs to be logged
    '''

    # Open the current logged_URL.json file and load its content
    with open("logged_URL.json", "r") as f:
        logged_URLs = json.load(f)

    # Get the latest logged URL index and then add the new URLs to the JSON structure
    latest_index = np.max([int(key) for key in logged_URLs[web_type].keys()]) if logged_URLs[web_type] else 1
    
    for i, link in enumerate(url[::-1]):  # Reverse the list to log from the bottom
        logged_URLs[web_type][str(latest_index + i)] = link

    # save the updated JSON structure back to the file
    with open("logged_URL.json", "w") as f:
        json.dump(logged_URLs, f, indent=4)


def get_past_URLs(web_type: Literal["Disnakerja", "Inginkerja", "RekrutmenBersama"] = None) -> str:
    '''
    Get the latest logged URL for each job website from the logged_URL.json file.
    '''

    # Check if logged_URL.json exists, if not create it and return None
    if not os.path.exists("logged_URL.json"):
        json_structure = {
            "Disnakerja": {
            },
            "Inginkerja": {
            },
            "RekrutmenBersama": {
            }
        }
        with open("logged_URL.json", "w") as f:
            json.dump(json_structure, f)
        return None

    # if it does exist, load the latest logged URL of the specified job website
    else:
        with open("logged_URL.json", "r") as f:
            logged_URLs = json.load(f)
        # Get the latest logged URL for each job website
        latest_url = logged_URLs[web_type].get(str(np.max([int(key) for key in logged_URLs[web_type].keys()]))) if logged_URLs[web_type] else None
        
        return latest_url
    

class base_job_scraper:
    def __init__(self, local_log: bool = True, discord_log: bool = False, 
                 website_loker: Literal["Disnakerja", "Inginkerja", "RekrutmenBersama"] = None):
        '''
        Initialize the base job scraper class.

        Parameters
        ----------
        - local_log : bool
            Whether to log the accepted vacancies to the local database (default: True)
        - discord_log : bool
            Whether to send the accepted vacancies to the Discord webhook (default: False)
        - website_loker : Literal["Disnakerja", "Inginkerja", "RekrutmenBersama"]
            The job website to scrape (default: None)
        '''
        # For logging purposes
        self.accepted_vacancies = []
        self.all_vacancies = []
        self.temp_logged_URLs = []

        self.website_loker = website_loker
        self.search_status = 1  # Status to control the scraping loop, 1 means continue, 0 means stop

        # Store the latest URL to avoid re-scraping already processed job postings
        self.latest_url = get_past_URLs(self.website_loker)

        # Logging preferences, local will log to the local database, discord will send a message to the discord webhook
        self.local_log = local_log
        self.discord_log = discord_log


    def get_vacancy_page(self):
        '''
        Scrape the job vacancy page and extract relevant information.

        This function will extract the company name, icon, view count, last updated date, location, experience needed, and job description from the job vacancy page.

        It will also verify if the vacancy is suitable for the applicant using the Gemini API and log the accepted vacancies to the local database and/or send them to the Discord webhook based on the logging preferences.
        '''
        # For a page
        content = WebDriverWait(DRIVER, 10).until(EC.presence_of_element_located((By.XPATH, "//div[@class = 'content-area']")))

        # Get the top content of the job listing
        top_content = content.find_element(By.XPATH, ".//header")
        icon = top_content.find_element(By.XPATH, ".//img").get_attribute("src")
        nama_perusahaan = top_content.find_element(By.XPATH, ".//h1[@itemprop='name']").text
        view_count = top_content.find_element(By.XPATH, ".//span[@class='gmr-view']").text

        # Get the lower content of the job listing
        lower_content = content.find_element(By.XPATH, ".//div[@class='row']")
        side_content = lower_content.find_element(By.XPATH, ".//div[@id='specs']")
        side_content_text = side_content.text
        jobdescription = lower_content.find_element(By.XPATH, ".//div[@id='description']").text
        side_content_parsed = [line.strip() for line in side_content_text.split('\n') if line.strip()]

        # Extracting specific details from the parsed side content based on the website
        match self.website_loker:
            case "Disnakerja":
                last_updated = side_content_parsed[1]
                location = side_content_parsed[6]
                experience_needed = side_content_parsed[12]

                # work_type = side_content_parsed[8]
                # required_education = side_content_parsed[10]
                # category = side_content_parsed[3]
            case "Inginkerja":
                last_updated = side_content_parsed[3]
                location = side_content_parsed[8]
                experience_needed = "No info"
            case "RekrutmenBersama":
                last_updated = side_content_parsed[1]
                location = side_content_parsed[7]
                experience_needed = "No info"
        
        # Stop if the job posting is older than the maximum allowed days
        last_updated_date_object = datetime.datetime.strptime(last_updated, "%B %d, %Y").date()
        if last_updated_date_object < MAX_PAST_DATE:
            self.search_status = 0
            return None
        
        # Verify if the vacancy is suitable for the applicant using the Gemini API
        Full_job_description = side_content_text + "\n" + jobdescription
        status = 0
        while not status:
            response, status = verify_vacancy(Full_job_description)
            if not status:
                time.sleep(API_TIMEOUT) # Try to get it indefinitely until the API is available.

        acceptance, position, reason = parse_json_response(response) # Safely parse the response to get acceptance, position, and reason from json the format

        # Change position column to a string, bcz it's a list
        position = ", ".join(position) if isinstance(position, list) else position

        # print(f"Reason: {reason}")

        # If the vacancy is accepted
        if acceptance == "1":
            returned_job = {
                "Nama_Perusahaan": nama_perusahaan,
                "Icon": icon, 
                "View_Count": view_count,
                "Last_Updated": last_updated,
                "Location": location,
                "Experience_Needed": experience_needed,
                "Position": position,
                "Reason": reason,
                "Post_link": DRIVER.current_url,
                "Website_Loker": self.website_loker
            }
            if self.discord_log:
                send_post_on_discord(returned_job)
                
            self.accepted_vacancies.append(returned_job)

        # Overall logs, doesn't matter if the vacancy is accepted or not, js log it
        overall_log = {
            "Nama_Perusahaan": nama_perusahaan,
            "View_Count": view_count,
            "Last_Updated": last_updated,
            "Location": location,
            "Experience_Needed": experience_needed,
            "Position": position,
            "Post_link": DRIVER.current_url,
            "Website_Loker": self.website_loker,
            "DESCRIPTION": Full_job_description,
        }

        self.all_vacancies.append(overall_log)

    def start(self, current_page: int = 1) -> tuple:
        '''
        Start the job scraping process for the specified job website.

        Parameters
        ----------
        - current_page : int
            The page number to start scraping from (default: 1)

        Returns
        -------
        - self.accepted_vacancies : list
            A list of accepted vacancies that match the applicant's profile
        - self.all_vacancies : list
            A list of all vacancies that were scraped, regardless of acceptance
        '''

        # Get the base URL for the specified job website
        Used_URL = WEB_DICT[self.website_loker]

        while self.search_status:
            DRIVER.get(Used_URL + str(current_page))

            # Wait for the job board to load
            try:
                job_board = WebDriverWait(DRIVER, LONG_WAIT).until(EC.presence_of_element_located((By.ID, "gmr-main-load")))
            except:
                job_board = None
                return self.accepted_vacancies, self.all_vacancies

            # Get all job listings on the current page and their links
            job_lists = job_board.find_elements(By.XPATH, "//article")
            job_link_lists = [job.find_element(By.XPATH, ".//h2/a").get_attribute("href") for job in job_lists]

            # Process each link
            for link in job_link_lists:
                if link == self.latest_url:
                    self.search_status = 0  # Stop the search if the latest URL is reached
                    break

                DRIVER.get(link)
                self.temp_logged_URLs.append(link)  # Log the URL for later use

                self.get_vacancy_page()

            current_page += 1

        # Logging processes
        log_URLs(self.website_loker, self.temp_logged_URLs) # Log the URLs that have been processed to avoid re-scraping in the future
        if self.local_log:
            insert_rows(self.accepted_vacancies, db_type="accepted")
            insert_rows(self.all_vacancies, db_type="overall")

        return self.accepted_vacancies, self.all_vacancies


def main(log_to_discord: bool = True, log_to_local: bool = True, report_error: bool = True):
    if report_error:
        EMAIL_CREDENTIALS = {
            "email": os.getenv("EMAIL"),
            "password": os.getenv("PASSWORD"),
            "send_to_email": os.getenv("SEND_TO_EMAIL")
        }
    if log_to_local:
        init_db()

    for website in list(WEB_DICT.keys()):
        try:
            theProcess = base_job_scraper(local_log=log_to_local, discord_log=log_to_discord, website_loker=website)
            theProcess.start()
        except Exception as exc:
            print(f"Error in main: {exc}")
            if report_error:
                send_email(str(exc), EMAIL_CREDENTIALS)

In [ ]:
# theProcess = base_job_scraper(local_log=True, discord_log=True, website_loker="Disnakerja")
# theProcess.start()

## Complete Debug